# BUỔI 04 — HADOOP HDFS VÀ THUẬT TOÁN MAPREDUCE

**Notebook thực hành dành cho Học viên**

| | |
|:---|:---|
| **Bộ dữ liệu** | `ai4i2020.csv` — tiếp nối trực tiếp từ Buổi 01 |
| **Thời lượng** | 60 phút (Lý thuyết 25' + Thực hành 35') |
| **Môi trường** | Cụm Hadoop 3.2.1 trên Docker |

---

## Buổi này nối tiếp Buổi 01 như thế nào

Ở Buổi 01, bạn xử lý `ai4i2020.csv` **trên một máy**, dữ liệu nằm gọn trong RAM. Cũng ở buổi đó, phép ngoại suy Volume cho thấy 50 máy chạy 24/7 trong một năm sẽ sinh ra **hàng chục GB** — vượt xa khả năng một máy đơn.

Hôm nay ta trả lời câu hỏi bỏ ngỏ đó:

```text
BUỔI 01                              BUỔI 04
Một máy, dữ liệu trong RAM     ──►   Nhiều máy, dữ liệu chia khối
pandas.read_csv()              ──►   hdfs dfs -put
df.groupby("Type")             ──►   MapReduce: Map → Shuffle → Reduce
L 3,92% · M 2,77% · H 2,09%    ══►   KẾT QUẢ PHẢI GIỐNG HỆT
```

> **Mục tiêu lớn nhất của buổi học:** cuối buổi, MapReduce phải cho ra **đúng ba con số mà Pandas đã cho ở Buổi 01**. Khi hai công nghệ hoàn toàn khác nhau hội tụ về cùng kết quả, bạn sẽ hiểu MapReduce không phải phép thuật — nó chỉ là `groupby` viết lại để chạy trên nhiều máy.

---

## Cách làm việc với notebook này

Mỗi nhiệm vụ có **4 thành phần**:

1. **Nhiệm vụ & Kết quả mong đợi**
2. **PROMPT CHO AI AGENT** — dán nguyên khối vào `⌘+I` / `Ctrl+I`
3. **GỢI Ý GIẢI** — khung code sẵn sàng dùng
4. **TỰ KIỂM TRA** — ô chấm điểm tự động

> **Trước khi bắt đầu, mở terminal chạy:**
> ```
> cd 00_Huong_Dan_Chung_va_Docker
> docker compose up -d hadoop-namenode hadoop-datanode
> ```
> Đợi 30–40 giây cho cả hai container đạt trạng thái `healthy`.

## Ô CÀI ĐẶT THƯ VIỆN — chạy một lần đầu tiên

Ô dưới dùng magic `%pip` (không phải `!pip`) để cài đúng vào Python kernel đang chạy notebook này.
Sau khi cài xong lần đầu, **khởi động lại kernel** (Kernel > Restart) rồi chạy tiếp các ô sau.


In [ ]:
# --- Cài thư viện vào đúng kernel đang chạy ---
# %pip (magic) cài vào sys.executable của kernel, khác với !pip (shell) có thể trỏ nhầm Python khác.
%pip install -q -r ../requirements.txt

# Kiểm tra kernel đang dùng Python nào
import sys
print("Kernel Python:", sys.executable)


---
## Ô THIẾT LẬP — CHẠY Ô NÀY ĐẦU TIÊN

Ô này dò đường dẫn dữ liệu, kiểm tra cụm Hadoop, và tạo hàm trợ giúp `hadoop()` để bạn chạy lệnh HDFS ngay trong notebook. **Không cần sửa gì.**

In [1]:
# =============================================================================
# Ô THIẾT LẬP - Chạy đầu tiên, không cần chỉnh sửa
# =============================================================================
import subprocess
import sys
import time
from pathlib import Path

NAMENODE   = "bigdata-hadoop-namenode"
HDFS_BASE  = "/user/bigdata/lab4"
HDFS_IN    = f"{HDFS_BASE}/input"
HDFS_OUT   = f"{HDFS_BASE}/output_mr"
STREAM_JAR = "$HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-3.2.1.jar"


def _tim_du_lieu():
    """Đi ngược cây thư mục tìm ai4i2020.csv trong kho dữ liệu dùng chung."""
    goc = Path.cwd().resolve()
    for tm in [goc, *goc.parents]:
        for uv in [tm / "00_Huong_Dan_Chung_va_Docker" / "data" / "raw" / "ai4i2020.csv",
                   tm / "data" / "raw" / "ai4i2020.csv"]:
            if uv.exists():
                return uv
    raise FileNotFoundError("Không tìm thấy ai4i2020.csv — hãy mở notebook từ thư mục dự án.")


DATA_PATH  = _tim_du_lieu()
OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)


def hadoop(lenh, hien_thi=True, im_lang_log=True):
    """Chạy một lệnh bên trong container Hadoop và trả về stdout.

    Ví dụ:  hadoop("hdfs dfs -ls /")
    """
    kq = subprocess.run(["docker", "exec", NAMENODE, "bash", "-c", lenh],
                        capture_output=True, text=True)
    out, err = kq.stdout, kq.stderr
    if im_lang_log:      # lọc bớt dòng INFO/WARN của Hadoop cho dễ đọc
        err = "\n".join(d for d in err.split("\n")
                        if d.strip() and not d.startswith(("20", "WARN", "SLF4J")))
    if hien_thi:
        if out.strip():
            print(out.rstrip())
        if err.strip():
            print("[thông báo]", err.rstrip(), file=sys.stderr)
    return out


def hdfs(doi_so, **kw):
    """Rút gọn cho `hdfs dfs`.   Ví dụ: hdfs("-ls -h /user")"""
    return hadoop(f"hdfs dfs {doi_so}", **kw)


# --- Kiểm tra môi trường ------------------------------------------------------
print("=" * 78)
print("MÔI TRƯỜNG THỰC HÀNH - BUỔI 04 (HADOOP HDFS & MAPREDUCE)")
print("=" * 78)
print(f"File dữ liệu    : {DATA_PATH}")
print(f"Dung lượng      : {DATA_PATH.stat().st_size / 1024**2:.3f} MB")
print(f"Thư mục kết quả : {OUTPUT_DIR}")
print("-" * 78)

_ps = subprocess.run(["docker", "ps", "--format", "{{.Names}}\t{{.Status}}"],
                     capture_output=True, text=True).stdout
_hadoop_rows = [d for d in _ps.split("\n") if "hadoop" in d.lower()]

if not _hadoop_rows:
    print("CHƯA THẤY CONTAINER HADOOP NÀO ĐANG CHẠY.")
    print("Mở terminal và chạy:")
    print("     cd 00_Huong_Dan_Chung_va_Docker")
    print("     docker compose up -d hadoop-namenode hadoop-datanode")
    print("Đợi 30-40 giây rồi chạy lại ô này.")
else:
    for d in _hadoop_rows:
        print("  ", d)
    print("-" * 78)
    _ver = hadoop("hadoop version | head -1", hien_thi=False).strip()
    print(f"  {_ver}")
    print("Giao diện web NameNode : http://localhost:9870")
    print("Giao diện web DataNode : http://localhost:9864")
    print("=" * 78)
    print("Sẵn sàng. Chuyển sang D1.")

FileNotFoundError: Không tìm thấy ai4i2020.csv — hãy mở notebook từ thư mục dự án.

---
---
# VÍ DỤ MẪU — LÀM MẪU TRƯỚC KHI TỰ LÀM

Phần này gồm **6 ví dụ mẫu chạy được ngay** (không có TODO). Mỗi ví dụ làm mẫu đúng một
kỹ thuật mà bài tập D1–D6 phía sau yêu cầu bạn tự viết lại trên bộ dữ liệu thật.

| Ví dụ | Làm mẫu cho | Nội dung |
|:--|:--|:---|
| VD1 | D1 | Bộ lệnh HDFS CLI cơ bản qua hàm `hadoop()` / `hdfs()` |
| VD2 | D2 | Vòng đời một tệp: `docker cp` → `-put` → `-cat` → `-rm` |
| VD3 | D3 | `getconf`, `-D dfs.blocksize`, đọc `fsck` |
| VD4 | D4 | MapReduce bằng Python thuần trên dữ liệu tí hon |
| VD5 | D5 | Hadoop Streaming với `awk` trên tệp mẫu |
| VD6 | D6 | Đối chiếu kết quả MapReduce với Pandas |

> Toàn bộ ví dụ dùng thư mục riêng `/user/bigdata/lab4/demo` và tệp `demo_sensor.csv`
> nên **không đụng chạm** gì tới dữ liệu bài tập trong `/user/bigdata/lab4/input`.
> Chạy VD0 dưới đây trước để tạo dữ liệu mẫu.

### VD0 — Tạo dữ liệu mẫu tí hon

12 dòng, 3 loại máy `L` / `M` / `H`, cột cuối `Machine failure` (1 = hỏng).
Nhỏ tới mức bạn **nhẩm tay đối chiếu được** — đó là lý do dùng nó để học thuật toán.

In [ ]:
# --- VD0: tạo dữ liệu mẫu trên máy thật ---
HDFS_DEMO = f"{HDFS_BASE}/demo"

DEMO_CSV = OUTPUT_DIR / "demo_sensor.csv"
DEMO_CSV.write_text(
    "UDI,Product ID,Type,Air temp,Process temp,Rot speed,Torque,Tool wear,Machine failure\n"
    "1,L001,L,298.1,308.6,1551,42.8,0,0\n"
    "2,L002,L,298.2,308.7,1408,46.3,3,0\n"
    "3,L003,L,298.1,308.5,1498,49.4,5,1\n"
    "4,M001,M,298.2,308.6,1433,39.5,7,0\n"
    "5,M002,M,298.1,308.7,1408,40.0,9,0\n"
    "6,M003,M,298.3,308.8,1425,41.9,11,1\n"
    "7,H001,H,298.5,309.0,1558,42.4,14,0\n"
    "8,H002,H,298.4,308.9,1527,40.2,16,0\n"
    "9,L004,L,298.6,309.1,1667,28.6,18,0\n"
    "10,L005,L,298.6,309.0,1741,28.0,21,1\n"
    "11,M004,M,298.6,309.1,1782,23.9,24,0\n"
    "12,H003,H,298.6,309.2,1423,44.3,29,0\n",
    encoding="utf-8",
)

print("Đã tạo:", DEMO_CSV)
print("Thư mục HDFS cho ví dụ:", HDFS_DEMO)
print("-" * 60)
print(DEMO_CSV.read_text(encoding="utf-8"))

# Kết quả đúng, tính tay để đối chiếu ở VD4/VD5/VD6:
#   L: 5 bản ghi, 2 hỏng -> 40,00%
#   M: 4 bản ghi, 1 hỏng -> 25,00%
#   H: 3 bản ghi, 0 hỏng ->  0,00%

### VD1 — Bộ lệnh HDFS CLI cơ bản  *(làm mẫu cho D1)*

`hadoop("...")` chạy một lệnh bất kỳ **bên trong container**; `hdfs("...")` là lối tắt
cho `hdfs dfs`. Cả hai đều **trả về** stdout dạng chuỗi nên bạn có thể xử lý tiếp bằng Python.

| Lệnh | Trả lời câu hỏi |
|:---|:---|
| `hdfs dfsadmin -report` | Cụm có mấy nút sống, còn bao nhiêu dung lượng? |
| `hdfs dfs -df -h` | Đã dùng bao nhiêu phần trăm dung lượng? |
| `hdfs dfs -ls -R /user` | Cây thư mục đang có những gì? |
| `hdfs version` | Bản Hadoop nào? |

In [ ]:
# --- VD1: các lệnh xem trạng thái cụm ---
print("=== ① PHIÊN BẢN HADOOP ===")
hadoop("hadoop version | head -2")

print("\n=== ② TÓM TẮT CỤM (5 dòng đầu của dfsadmin -report) ===")
hadoop("hdfs dfsadmin -report | head -5")

print("\n=== ③ DUNG LƯỢNG ===")
hdfs("-df -h")

print("\n=== ④ CÂY THƯ MỤC /user (nếu chưa có gì thì rỗng là bình thường) ===")
hdfs("-ls -R /user 2>/dev/null || echo '(chưa có /user)'")

# Bắt giá trị về Python thay vì chỉ in ra màn hình:
bao_cao = hadoop("hdfs dfsadmin -report", hien_thi=False)
import re
so_nut = re.search(r"Live datanodes \((\d+)\)", bao_cao)
print("\n=== ⑤ TÁCH SỐ LIỆU BẰNG PYTHON ===")
print("Số DataNode đang sống:", so_nut.group(1) if so_nut else "không đọc được")

### VD2 — Vòng đời một tệp trên HDFS  *(làm mẫu cho D2)*

Nhớ **hai bước bắt buộc**: lệnh `hdfs` chỉ tồn tại bên trong container, nên tệp phải đi
qua container trước.

```text
máy thật  ──① docker cp──►  container  ──② hdfs dfs -put──►  HDFS
```

Ví dụ này đi trọn vòng đời: đưa lên → liệt kê → đọc → đếm dòng → tải về → xóa.

In [ ]:
# --- VD2: put / ls / cat / get / rm ---
# ① Chép từ máy thật vào container
subprocess.run(["docker", "cp", str(DEMO_CSV), f"{NAMENODE}:/tmp/demo_sensor.csv"], check=True)
print("① docker cp xong")

# ② Tạo thư mục và đẩy lên HDFS  (-f: ghi đè nếu chạy lại ô này)
hdfs(f"-mkdir -p {HDFS_DEMO}")
hdfs(f"-put -f /tmp/demo_sensor.csv {HDFS_DEMO}/")
print("② -put xong")

# ③ Liệt kê kèm kích thước dễ đọc
print("\n=== LIỆT KÊ ===")
hdfs(f"-ls -h {HDFS_DEMO}/")

# ④ Đọc nội dung: -cat (toàn bộ), -head (đầu tệp), -tail (cuối tệp)
print("\n=== 3 DÒNG ĐẦU ===")
hadoop(f"hdfs dfs -cat {HDFS_DEMO}/demo_sensor.csv 2>/dev/null | head -3")

# ⑤ Đếm dòng và xem thống kê thư mục
so_dong = hadoop(f"hdfs dfs -cat {HDFS_DEMO}/demo_sensor.csv 2>/dev/null | wc -l",
                 hien_thi=False).strip()
print(f"\nSố dòng trên HDFS: {so_dong}   (kỳ vọng 13 = 12 bản ghi + 1 tiêu đề)")

print("\n=== -du -h (dung lượng)  và  -count (thư mục/tệp/byte) ===")
hdfs(f"-du -h {HDFS_DEMO}/")
hdfs(f"-count {HDFS_DEMO}")

# ⑥ Tải ngược từ HDFS về container rồi kiểm tra
hadoop(f"rm -f /tmp/tai_ve.csv && hdfs dfs -get {HDFS_DEMO}/demo_sensor.csv /tmp/tai_ve.csv")
hadoop("wc -l /tmp/tai_ve.csv")

# ⑦ Dọn dẹp minh họa: tạo tệp thừa rồi xóa nó đi
hdfs(f"-cp {HDFS_DEMO}/demo_sensor.csv {HDFS_DEMO}/ban_sao.csv")
print("\n=== TRƯỚC KHI XÓA ===");  hdfs(f"-ls {HDFS_DEMO}/")
hdfs(f"-rm -f -skipTrash {HDFS_DEMO}/ban_sao.csv", hien_thi=False)
print("=== SAU KHI XÓA ===");     hdfs(f"-ls {HDFS_DEMO}/")

### VD3 — Khối và bản sao  *(làm mẫu cho D3)*

Tệp mẫu chỉ vài trăm byte nên nằm gọn trong **một khối**. Muốn *nhìn thấy* việc chia khối,
ta nhân bản tệp cho to lên rồi **hạ kích thước khối xuống 1 MB** khi tải lên.

Công thức số khối:

```text
số khối = ceil( kích thước tệp / kích thước khối )
khối cuối cùng gần như luôn KHÔNG đầy — HDFS không đệm thêm byte rác
```

In [ ]:
# --- VD3: getconf, -D dfs.blocksize, fsck ---
# ① Cấu hình mặc định của cụm
bs = int(hadoop("hdfs getconf -confKey dfs.blocksize", hien_thi=False).strip())
rp = hadoop("hdfs getconf -confKey dfs.replication", hien_thi=False).strip()
print(f"dfs.blocksize   = {bs:,} byte = {bs/1024**2:.0f} MB")
print(f"dfs.replication = {rp}")

# ② fsck tệp nhỏ -> đúng 1 khối
print("\n=== FSCK TỆP MẪU (vài trăm byte) ===")
hadoop(f"hdfs fsck {HDFS_DEMO}/demo_sensor.csv -files -blocks 2>/dev/null | head -6")

# ③ Nhân tệp mẫu lên ~2,5 MB ngay trong container
hadoop("for i in $(seq 1 4000); do cat /tmp/demo_sensor.csv; done > /tmp/demo_big.csv")
hadoop("ls -l /tmp/demo_big.csv | awk '{print \"Kích thước tệp lớn:\", $5, \"byte\"}'")

# ④ Đẩy lên với khối 1 MB (tham số -D đặt TRƯỚC lệnh con -put)
hdfs(f"-D dfs.blocksize=1048576 -put -f /tmp/demo_big.csv {HDFS_DEMO}/", hien_thi=False)

# ⑤ fsck lại -> nhiều khối, khối cuối không đầy
print("\n=== FSCK TỆP 2,5 MB VỚI KHỐI 1 MB ===")
fs = hadoop(f"hdfs fsck {HDFS_DEMO}/demo_big.csv -files -blocks", hien_thi=False)
import re
for dong in fs.split("\n"):
    if "blk_" in dong or "block(s)" in dong:
        print("   ", dong.strip())

lens = [int(x) for x in re.findall(r"len=(\d+)", fs)]
print("\nSố khối       :", len(lens))
for i, L in enumerate(lens):
    day = "ĐẦY" if L == 1048576 else "KHÔNG đầy  ◄── phần dư"
    print(f"   khối {i}: {L:>9,} byte   {day}")
print("Tổng byte     :", f"{sum(lens):,}")

### VD4 — MapReduce bằng Python thuần  *(làm mẫu cho D4)*

Bốn giai đoạn, chạy trên 12 dòng để bạn **nhìn thấy từng cặp khóa–giá trị**:

```text
① SPLIT    12 dòng            →  2 mảnh × 6 dòng
② MAP      mỗi dòng           →  cặp (Type, Machine failure)
③ SHUFFLE  gom theo khóa      →  3 khóa: H, L, M
④ REDUCE   tổng hợp mỗi khóa  →  3 dòng kết quả
```

Nguyên tắc cốt lõi: **Map chạy độc lập trên từng mảnh** (nên song song hóa được),
còn **Reduce chỉ chạy sau khi cùng một khóa đã được gom về một chỗ**.

In [ ]:
# --- VD4: mô phỏng MapReduce, in đủ dữ liệu trung gian ---
import csv
from collections import defaultdict

# ========== ① SPLIT ==========
with open(DEMO_CSV, encoding="utf-8") as f:
    tat_ca = list(csv.reader(f))
tieu_de, du_lieu = tat_ca[0], tat_ca[1:]

SO_MANH = 2
kt = len(du_lieu) // SO_MANH
manh = [du_lieu[i:i + kt] for i in range(0, len(du_lieu), kt)]

print("① SPLIT")
for i, m in enumerate(manh):
    print(f"   Mảnh {i}: {len(m)} dòng — UDI {m[0][0]}..{m[-1][0]}")

# ========== ② MAP ==========
def mapper(mang_dong):
    """Mỗi dòng -> một cặp (khóa, giá trị) = (Type, Machine failure)."""
    return [(c[2], int(c[8])) for c in mang_dong if len(c) >= 9]

ket_qua_map = [mapper(m) for m in manh]

print("\n② MAP  (mỗi mảnh xử lý độc lập — đây là chỗ Hadoop chạy song song)")
for i, kq in enumerate(ket_qua_map):
    print(f"   Mảnh {i} -> {kq}")

# ========== ③ SHUFFLE ==========
gom = defaultdict(list)
for kq in ket_qua_map:
    for khoa, gia_tri in kq:
        gom[khoa].append(gia_tri)
gom = dict(sorted(gom.items()))

print("\n③ SHUFFLE & SORT  (gom mọi giá trị cùng khóa về một nơi)")
for khoa, gia_tri in gom.items():
    print(f"   {khoa} -> {gia_tri}")

# ========== ④ REDUCE ==========
def reducer(danh_sach):
    tong = len(danh_sach)
    hong = sum(danh_sach)
    return tong, hong, hong / tong * 100

ket_qua_vd4 = {k: reducer(v) for k, v in gom.items()}

print("\n④ REDUCE")
print(f"   {'Type':<6}{'Bản ghi':>9}{'Hỏng':>7}{'Tỷ lệ':>9}")
print("   " + "-" * 31)
for k, (tong, hong, ty_le) in ket_qua_vd4.items():
    print(f"   {k:<6}{tong:>9}{hong:>7}{ty_le:>8.2f}%")
print("\n   Nhẩm tay đối chiếu: L 5/2 = 40,00% · M 4/1 = 25,00% · H 3/0 = 0,00%")

### VD5 — Hadoop Streaming với `awk`  *(làm mẫu cho D5)*

Hadoop Streaming cho phép Mapper/Reducer là **chương trình bất kỳ** đọc stdin, ghi stdout.
Container khóa học không có Python nên ta dùng `awk`.

```text
CSV ──► [stdin] mapper.sh [stdout] "Type \t failure" ──► Hadoop sort ──► [stdin] reducer.sh ──► kết quả
```

**Luôn TEST bằng ống dẫn Linux trước khi nộp job** — chạy trong một giây thay vì chờ job
mất 10–30 giây rồi mới báo lỗi:

```bash
cat file | mapper.sh | sort | reducer.sh
```

In [ ]:
# --- VD5: Hadoop Streaming trên tệp mẫu ---
HDFS_DEMO_OUT = f"{HDFS_BASE}/demo_output"

MAPPER_VD = r"""#!/bin/bash
# MAP: bỏ dòng tiêu đề, in ra cặp <Type \t Machine failure>
awk -F, '$1 != "UDI" { print $3 "\t" $9 }'
"""

REDUCER_VD = r"""#!/bin/bash
# REDUCE: cộng dồn theo Type
awk -F"\t" '
{ tong[$1]++; hong[$1] += $2 }
END { for (t in tong) printf "%s\t%d\t%d\t%.2f%%\n", t, tong[t], hong[t], hong[t]*100/tong[t] }
'
"""

# ① Ghi hai script vào container (heredoc trích dẫn 'HET' để bash không diễn giải $ và \)
for ten, noi_dung in [("vd_mapper.sh", MAPPER_VD), ("vd_reducer.sh", REDUCER_VD)]:
    hadoop(f"cat > /tmp/{ten} <<'HET'\n{noi_dung}HET\nchmod +x /tmp/{ten}", hien_thi=False)
    print("① đã tạo /tmp/" + ten)

# ② TEST bằng ống dẫn Linux — nhanh, không cần cụm
print("\n② TEST CỤC BỘ (cat | mapper | sort | reducer)")
print("   -- đầu ra của MAPPER (5 cặp đầu):")
hadoop("cat /tmp/demo_sensor.csv | /tmp/vd_mapper.sh | head -5")
print("   -- đầu ra của REDUCER:")
hadoop("cat /tmp/demo_sensor.csv | /tmp/vd_mapper.sh | sort | /tmp/vd_reducer.sh")

# ③ Xóa output cũ — Hadoop TỪ CHỐI chạy nếu thư mục đích đã tồn tại
hdfs(f"-rm -r -f -skipTrash {HDFS_DEMO_OUT}", hien_thi=False)
print("\n③ đã dọn thư mục đầu ra cũ")

# ④ Nộp job thật lên cụm
print("\n④ ĐANG CHẠY JOB (khoảng 10-30 giây)...")
t0 = time.perf_counter()
log = hadoop(
    f"hadoop jar {STREAM_JAR} "
    f"-D mapreduce.job.name='VD5_demo_type_failure' "
    f"-files /tmp/vd_mapper.sh,/tmp/vd_reducer.sh "
    f"-input {HDFS_DEMO}/demo_sensor.csv "
    f"-output {HDFS_DEMO_OUT} "
    f"-mapper vd_mapper.sh -reducer vd_reducer.sh 2>&1",
    hien_thi=False,
)
t_job = time.perf_counter() - t0
print(f"   xong sau {t_job:.1f} giây")

print("\n   -- Bộ đếm quan trọng trong log:")
for dong in log.split("\n"):
    if any(k in dong for k in ("Map input records", "Map output records",
                               "Reduce input records", "Reduce output records",
                               "Job job_", "completed successfully")):
        print("     ", dong.strip())

# ⑤ Đọc kết quả
print("\n⑤ KẾT QUẢ TRÊN HDFS")
hdfs(f"-ls {HDFS_DEMO_OUT}")
print("   -- nội dung part-*:")
hadoop(f"hdfs dfs -cat {HDFS_DEMO_OUT}/part-* 2>/dev/null")
print("   (phải trùng khớp với VD4: L 40,00% · M 25,00% · H 0,00%)")

### VD6 — Đối chiếu với Pandas  *(làm mẫu cho D6)*

Cùng một phép tính, ba đường đi khác nhau: Python thuần (VD4), Hadoop (VD5), Pandas (VD6).
**Kết quả phải giống hệt nhau** — nếu lệch, gần như chắc chắn Mapper cắt nhầm cột.

Điều đáng suy nghĩ: trên 12 dòng thì Pandas nhanh hơn Hadoop cả nghìn lần. Hadoop chỉ
thắng khi dữ liệu **không nhét vừa RAM một máy** — chi phí khởi động job là cố định,
còn Pandas thì tăng tuyến tính rồi sập khi hết RAM.

In [ ]:
# --- VD6: pandas groupby + đối chiếu ba cách làm ---
import pandas as pd

t0 = time.perf_counter()
df_demo = pd.read_csv(DEMO_CSV)
kq_pandas = (df_demo.groupby("Type")
             .agg(so_ban_ghi=("Machine failure", "size"),
                  so_lan_hong=("Machine failure", "sum"))
             .assign(ty_le_hong=lambda d: d.so_lan_hong / d.so_ban_ghi * 100))
t_pandas_ms = (time.perf_counter() - t0) * 1000

print("=== PANDAS ===")
print(kq_pandas.round(2))
print(f"\nThời gian Pandas: {t_pandas_ms:.1f} ms")

# Đối chiếu Pandas với kết quả MapReduce thủ công ở VD4
print("\n=== ĐỐI CHIẾU VD4 (Python) ↔ VD6 (Pandas) ===")
print(f"{'Type':<6}{'MR bản ghi':>12}{'PD bản ghi':>12}{'MR hỏng':>10}{'PD hỏng':>10}   Khớp")
print("-" * 62)
tat_ca_khop = True
for t in sorted(ket_qua_vd4):
    mr_tong, mr_hong = ket_qua_vd4[t][0], ket_qua_vd4[t][1]
    pd_tong = int(kq_pandas.loc[t, "so_ban_ghi"])
    pd_hong = int(kq_pandas.loc[t, "so_lan_hong"])
    khop = (mr_tong, mr_hong) == (pd_tong, pd_hong)
    tat_ca_khop &= khop
    print(f"{t:<6}{mr_tong:>12}{pd_tong:>12}{mr_hong:>10}{pd_hong:>10}   {'có' if khop else 'KHÔNG'}")

print("-" * 62)
print("Ba cách làm cho cùng một kết quả." if tat_ca_khop else "Có sai lệch — kiểm tra lại chỉ số cột trong mapper.")

try:
    print(f"\nPandas {t_pandas_ms:.1f} ms  ·  Job Hadoop {t_job:.1f} s "
          f"→ Pandas nhanh hơn khoảng {t_job*1000/t_pandas_ms:,.0f} lần trên 12 dòng dữ liệu.")
except NameError:
    print("\n(Chạy VD5 trước để có thời gian job Hadoop mà so sánh.)")

### VD7 — Dọn dẹp phần ví dụ *(tùy chọn)*

Chạy ô này nếu muốn xóa hết dấu vết của phần ví dụ trên HDFS trước khi làm bài tập.
Dữ liệu bài tập trong `/user/bigdata/lab4/input` **không bị ảnh hưởng**.

In [ ]:
# --- VD7: dọn dẹp (tùy chọn) ---
hdfs(f"-rm -r -f -skipTrash {HDFS_DEMO} {HDFS_BASE}/demo_output", hien_thi=False)
hadoop("rm -f /tmp/demo_sensor.csv /tmp/demo_big.csv /tmp/tai_ve.csv "
       "/tmp/vd_mapper.sh /tmp/vd_reducer.sh", hien_thi=False)
print("Đã dọn thư mục ví dụ trên HDFS và các tệp tạm trong container.")
print("Còn lại dưới", HDFS_BASE, ":")
hdfs(f"-ls {HDFS_BASE} 2>/dev/null || echo '  (trống)'")

---
---
# D1. KIỂM TRA SỨC KHỎE CỤM HADOOP
### 5 phút

### Nhiệm vụ

Xác nhận cụm đang chạy, đọc báo cáo trạng thái, và mở giao diện web NameNode.

### Kết quả mong đợi

Báo cáo hiển thị **`Live datanodes (1)`** và dung lượng khả dụng lớn hơn 0.

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Kiểm tra sức khỏe cụm Hadoop bằng hàm hadoop() đã có sẵn:
1. Chạy "hdfs dfsadmin -report" và in ra 15 dòng đầu
2. Chạy "hdfs dfs -df -h" để xem dung lượng cụm
3. Chạy "hdfs dfs -ls /" để xem thư mục gốc HDFS
In kết quả có tiêu đề rõ ràng cho từng phần.
```

### Công cụ

`hadoop()` · `hdfs dfsadmin -report` · `hdfs dfs -df -h` · `hdfs dfs -ls`

#### GỢI Ý GIẢI — D1

**Khung code:**
```python
print("=== BÁO CÁO TRẠNG THÁI CỤM ===")
hadoop("hdfs dfsadmin -report | head -15")

print("\n=== DUNG LƯỢNG CỤM ===")
hdfs("-df -h")

print("\n=== THƯ MỤC GỐC HDFS ===")
hdfs("-ls /")
```

**Cách đọc báo cáo `dfsadmin -report`:**

| Dòng | Ý nghĩa |
|:---|:---|
| `Configured Capacity` | Tổng dung lượng cụm nhìn thấy |
| `Present Capacity` | Dung lượng thực sự dùng được |
| `DFS Used` | Dữ liệu HDFS đang chiếm |
| `Live datanodes (N)` | **Số nút lưu trữ còn sống** — quan trọng nhất |
| `Under replicated blocks` | Số khối chưa đủ bản sao (phải bằng 0) |

> **Việc cần làm trên trình duyệt:** mở **http://localhost:9870**
> - Tab **Overview** — số nút sống, dung lượng, thời gian hoạt động
> - Tab **Datanodes** — danh sách nút lưu trữ
> - Tab **Utilities → Browse the file system** — duyệt cây thư mục HDFS (lát nữa sẽ thấy tệp của bạn ở đây)

> **Câu hỏi:** Vì sao NameNode có web UI riêng mà DataNode cũng có? Vì chúng là **hai tiến trình độc lập chạy trên hai máy khác nhau** trong cụm thật. Trong Docker chúng chỉ tình cờ cùng nằm trên một máy.

In [ ]:
# --- D1: Kiểm tra sức khỏe cụm ---
# TODO: In báo cáo dfsadmin -report (15 dòng đầu)
# TODO: In dung lượng cụm bằng -df -h
# TODO: Liệt kê thư mục gốc HDFS

In [ ]:
# =============================================================================
# TỰ KIỂM TRA D1 (không cần sửa)
# =============================================================================
def _kiem_tra_d1():
    bc = hadoop("hdfs dfsadmin -report", hien_thi=False)
    if not bc.strip():
        return print("Không lấy được báo cáo. Cụm Hadoop có đang chạy không?")

    import re
    m = re.search(r"Live datanodes \((\d+)\)", bc)
    song = int(m.group(1)) if m else 0
    cap = re.search(r"Present Capacity:\s+(\d+)", bc)
    gb = int(cap.group(1)) / 1024**3 if cap else 0

    print(f"Live datanodes      : {song}      (kỳ vọng ≥ 1)")
    print(f"Dung lượng khả dụng : {gb:.2f} GB")
    print("-" * 60)
    if song >= 1 and gb > 0:
        print("Cụm HDFS khỏe mạnh, sẵn sàng nhận dữ liệu.")
    else:
        print("Cụm chưa sẵn sàng. Kiểm tra: docker compose ps")
        print("Nếu vừa khởi động, NameNode có thể còn ở safe mode — đợi 30-60 giây.")

_kiem_tra_d1()

---
# D2. ĐƯA DỮ LIỆU LÊN HDFS
### 6 phút

### Nhiệm vụ

Tạo cây thư mục trên HDFS, đẩy `ai4i2020.csv` lên, rồi kiểm chứng bằng CLI.

### Quy trình BẮT BUỘC hai bước — điểm hay nhầm nhất

```text
Máy thật của bạn          Container Hadoop           HDFS
      │                          │                     │
      │  ① docker cp             │                     │
      ├─────────────────────────►│                     │
      │                          │  ② hdfs dfs -put    │
      │                          ├────────────────────►│
```

Phải qua **hai bước** vì lệnh `hdfs` chỉ tồn tại **bên trong** container. Bước ① đưa tệp vào container, bước ② mới đẩy lên HDFS.

### Kết quả mong đợi

```text
-rw-r--r--   1 root supergroup    500.0 K ... /user/bigdata/lab4/input/ai4i2020.csv
```
Và `wc -l` trên HDFS phải ra đúng **10001** (10.000 dòng + 1 tiêu đề).

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Đưa file dữ liệu lên HDFS, dùng các biến có sẵn DATA_PATH, NAMENODE, HDFS_IN
và hàm hadoop(), hdfs():
1. Dùng subprocess chạy "docker cp <DATA_PATH> <NAMENODE>:/tmp/" để chép file
   từ máy thật vào container
2. Tạo thư mục HDFS_IN trên HDFS bằng "hdfs dfs -mkdir -p"
3. Đẩy file từ /tmp trong container lên HDFS_IN bằng "hdfs dfs -put -f"
4. Kiểm chứng: liệt kê bằng "-ls -h", xem dung lượng bằng "-du -h",
   và đếm số dòng bằng "hdfs dfs -cat ... | wc -l"
In tiêu đề rõ ràng cho từng bước.
```

### Công cụ

`subprocess.run()` · `docker cp` · `hdfs dfs -mkdir -p` · `-put -f` · `-ls -h` · `-du -h` · `-cat`

#### GỢI Ý GIẢI — D2

**Khung code:**
```python
# ① Chép file từ máy thật vào container
subprocess.run(["docker", "cp", str(DATA_PATH), f"{NAMENODE}:/tmp/ai4i2020.csv"], check=True)
print("① Đã chép file vào container")

# ② Tạo thư mục trên HDFS
hdfs(f"-mkdir -p {HDFS_IN}")
print("② Đã tạo thư mục HDFS:", HDFS_IN)

# ③ Đẩy file lên HDFS  (-f để ghi đè nếu chạy lại)
hdfs(f"-put -f /tmp/ai4i2020.csv {HDFS_IN}/")
print("③ Đã đẩy file lên HDFS")

# ④ Kiểm chứng
print("\n=== LIỆT KÊ ===")
hdfs(f"-ls -h {HDFS_IN}/")

print("\n=== DUNG LƯỢNG ===")
hdfs(f"-du -h {HDFS_IN}/")

print("\n=== 3 DÒNG ĐẦU ===")
hadoop(f"hdfs dfs -cat {HDFS_IN}/ai4i2020.csv 2>/dev/null | head -3")

print("\n=== ĐẾM SỐ DÒNG ===")
so_dong = hadoop(f"hdfs dfs -cat {HDFS_IN}/ai4i2020.csv 2>/dev/null | wc -l",
                 hien_thi=False).strip()
print(f"Số dòng trên HDFS: {so_dong}   (kỳ vọng 10001)")
```

**Đọc kết quả `-ls -h`:**
```text
-rw-r--r--   1   root  supergroup   500.0 K   2026-08-24 17:40   /user/.../ai4i2020.csv
    │        │    │        │           │
    │        │    │        │           └─ dung lượng thật (khối cuối KHÔNG bị lấp đầy)
    │        │    │        └───────────── nhóm sở hữu
    │        │    └────────────────────── người sở hữu
    │        └─────────────────────────── SỐ BẢN SAO (replication) ◄── để ý con số này
    └──────────────────────────────────── quyền truy cập, giống Linux
```

> **Ba lỗi hay gặp:**
> 1. **Nhầm chiều `-put` / `-get`.** Mẹo nhớ: **put** = *đặt vào* HDFS, **get** = *lấy ra* khỏi HDFS.
> 2. **`put: File exists`** — chạy lần hai mà quên cờ `-f`. Thêm `-f` để ghi đè.
> 3. **`cat: Unable to write to output stream`** — do `head` đóng ống dẫn sớm. **Không phải lỗi**, kết quả vẫn đúng.

In [ ]:
# --- D2: Đưa dữ liệu lên HDFS ---
# TODO ①: docker cp file từ máy thật vào container (dùng subprocess.run)
# TODO ②: hdfs dfs -mkdir -p tạo thư mục HDFS_IN
# TODO ③: hdfs dfs -put -f đẩy file lên HDFS
# TODO ④: kiểm chứng bằng -ls -h, -du -h, và đếm dòng bằng -cat | wc -l

In [ ]:
# =============================================================================
# TỰ KIỂM TRA D2 (không cần sửa)
# =============================================================================
def _kiem_tra_d2():
    ds = hadoop(f"hdfs dfs -ls {HDFS_IN}/ai4i2020.csv", hien_thi=False)
    if "ai4i2020.csv" not in ds:
        return print(f"Chưa thấy file trên HDFS tại {HDFS_IN}/ai4i2020.csv")

    so_dong = hadoop(f"hdfs dfs -cat {HDFS_IN}/ai4i2020.csv 2>/dev/null | wc -l",
                     hien_thi=False).strip()
    phan = ds.split()
    ban_sao, kich_thuoc = phan[1], int(phan[4])

    print("File trên HDFS   : có mặt")
    print(f"Kích thước       : {kich_thuoc:,} byte  ({kich_thuoc/1024:.1f} KB)")
    print(f"Số bản sao       : {ban_sao}   (cụm này chỉ có 1 DataNode nên = 1)")
    print(f"Số dòng          : {so_dong}   (kỳ vọng 10001)")
    print("-" * 62)
    if so_dong == "10001":
        print("Dữ liệu đã lên HDFS nguyên vẹn, đủ 10.000 dòng + 1 dòng tiêu đề.")
        print("Mở http://localhost:9870 → Utilities → Browse the file system")
        print(f"   rồi vào {HDFS_IN} để nhìn thấy file bằng giao diện web.")
    else:
        print(f"Số dòng không khớp (được {so_dong}, cần 10001). Thử -put -f lại.")

_kiem_tra_d2()

---
# D3. KHÁM PHÁ CƠ CHẾ CHIA KHỐI
### 4 phút ·**TRỌNG TÂM PHẦN HDFS**

### Nhiệm vụ

Tệp 500 KB nhỏ hơn khối 128 MB rất nhiều nên chỉ có **1 khối** — chưa thấy gì thú vị. Vì vậy ta sẽ **tự tạo tệp lớn hơn với khối nhỏ hơn** để nhìn thấy hiện tượng chia khối thật sự.

| Bước | Việc làm |
|:---|:---|
| 1 | Đọc cấu hình mặc định `dfs.blocksize` và `dfs.replication` |
| 2 | `fsck` tệp hiện tại → thấy 1 khối |
| 3 | Nhân bản tệp 10 lần (~4,9 MB), tải lên với khối **1 MB** |
| 4 | `fsck` lại → thấy **5 khối** |

### Kết quả mong đợi

```text
/user/bigdata/lab4/input/ai4i_big.csv 5120440 bytes, replication=1, 5 block(s):  OK
0. blk_...1003 len=1048576 Live_repl=1
1. blk_...1004 len=1048576 Live_repl=1
2. blk_...1005 len=1048576 Live_repl=1
3. blk_...1006 len=1048576 Live_repl=1
4. blk_...1007 len=926136  Live_repl=1     ◄── khối cuối KHÔNG đầy
```

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Khám phá cơ chế chia khối của HDFS, dùng hàm hadoop() và hdfs() có sẵn:
1. Đọc 2 tham số cấu hình bằng "hdfs getconf -confKey dfs.blocksize"
   và "hdfs getconf -confKey dfs.replication". Đổi blocksize ra MB rồi in.
2. Chạy "hdfs fsck <đường dẫn file> -files -blocks" cho file ai4i2020.csv
   và nhận xét số khối thu được.
3. Tạo file lớn: chạy trong container vòng lặp nối file ai4i2020.csv 10 lần
   thành /tmp/ai4i_big.csv
4. Đẩy lên HDFS với kích thước khối 1MB:
   hdfs dfs -D dfs.blocksize=1048576 -put -f /tmp/ai4i_big.csv <thư mục input>
5. Chạy fsck lại cho file lớn, in ra danh sách khối và nhận xét
   khối cuối cùng có kích thước khác các khối trước.
```

### Công cụ

`hdfs getconf -confKey` · `hdfs fsck -files -blocks` · `hdfs dfs -D dfs.blocksize=...`

#### GỢI Ý GIẢI — D3

**Khung code:**
```python
# ① Cấu hình mặc định
bs = int(hadoop("hdfs getconf -confKey dfs.blocksize", hien_thi=False).strip())
rp = hadoop("hdfs getconf -confKey dfs.replication", hien_thi=False).strip()
print(f"dfs.blocksize   = {bs:,} byte = {bs/1024**2:.0f} MB")
print(f"dfs.replication = {rp}")

# ② fsck file hiện tại -> chỉ 1 khối
print("\n=== FSCK FILE 500 KB ===")
hadoop(f"hdfs fsck {HDFS_IN}/ai4i2020.csv -files -blocks 2>/dev/null | head -8")

# ③ Tạo file lớn ~4,9 MB bằng cách nối 10 lần
hadoop("for i in $(seq 1 10); do cat /tmp/ai4i2020.csv; done > /tmp/ai4i_big.csv")
hadoop("ls -lh /tmp/ai4i_big.csv | awk '{print $5}'")

# ④ Đẩy lên với khối chỉ 1 MB
hdfs(f"-D dfs.blocksize=1048576 -put -f /tmp/ai4i_big.csv {HDFS_IN}/")

# ⑤ fsck lại -> thấy nhiều khối
print("\n=== FSCK FILE 4,9 MB VỚI KHỐI 1 MB ===")
hadoop(f"hdfs fsck {HDFS_IN}/ai4i_big.csv -files -blocks 2>/dev/null "
       f"| grep -E '^/user|len=|Total blocks'")
```

**Ba điều PHẢI rút ra từ kết quả:**

1. **Tệp 5.120.440 byte chia thành 5 khối**, không phải một khối liền mạch.
2. **Bốn khối đầu đúng 1.048.576 byte** (= 1 MB), **khối cuối chỉ 926.136 byte**. Khối cuối *không bị lấp đầy* — nó chỉ chiếm đúng phần dữ liệu còn lại. Đây là điều học viên hay hiểu nhầm nhất: tệp 500 KB **không** chiếm trọn 128 MB trên đĩa.
3. Trong cụm thật có nhiều DataNode, **5 khối này nằm trên 5 máy khác nhau**, và 5 Mapper chạy song song — mỗi Mapper xử lý khối nằm sẵn trên máy mình. Đó chính là **Data Locality**: *di chuyển phép tính đến chỗ dữ liệu, không chuyển dữ liệu đến chỗ phép tính*.

> **Câu hỏi mở rộng:** `dfs.blocksize` nhỏ nhất HDFS chấp nhận là bao nhiêu?
>
> Mặc định là **1 MB** (`dfs.namenode.fs-limits.min-block-size`). Đặt nhỏ hơn sẽ bị từ chối, vì mỗi khối tốn ~150 byte siêu dữ liệu trong **RAM của NameNode**. Hàng triệu tệp nhỏ sẽ làm NameNode cạn RAM — vấn đề kinh điển tên là **"small files problem"**.

In [ ]:
# --- D3: Khám phá cơ chế chia khối ---
# TODO ①: Đọc dfs.blocksize và dfs.replication bằng hdfs getconf -confKey
# TODO ②: fsck file ai4i2020.csv -> quan sát số khối
# TODO ③: Tạo /tmp/ai4i_big.csv bằng cách nối file gốc 10 lần
# TODO ④: Đẩy lên HDFS với -D dfs.blocksize=1048576
# TODO ⑤: fsck lại file lớn -> đếm số khối, chú ý kích thước khối CUỐI

In [ ]:
# =============================================================================
# TỰ KIỂM TRA D3 (không cần sửa)
# =============================================================================
def _kiem_tra_d3():
    import re
    fs = hadoop(f"hdfs fsck {HDFS_IN}/ai4i_big.csv -files -blocks", hien_thi=False)
    if "ai4i_big.csv" not in fs:
        return print("Luu y:  Chưa tạo file ai4i_big.csv trên HDFS. Hãy hoàn thành bước ③ và ④.")

    m = re.search(r"(\d+) bytes.*?(\d+) block\(s\)", fs, re.S)
    lens = [int(x) for x in re.findall(r"len=(\d+)", fs)]
    if not m or not lens:
        return print("Luu y:  Không đọc được thông tin khối từ fsck.")

    tong_byte, so_khoi = int(m.group(1)), int(m.group(2))
    print(f"Kích thước file : {tong_byte:,} byte  ({tong_byte/1024**2:.2f} MB)")
    print(f"Số khối         : {so_khoi}")
    print("-" * 66)
    for i, L in enumerate(lens):
        nhan = "◄── khối CUỐI, không bị lấp đầy" if i == len(lens) - 1 and L != lens[0] else ""
        print(f"Khối {i}: {L:>9,} byte  ({L/1024**2:.3f} MB) {nhan}")
    print("-" * 66)
    print(f"Tổng các khối   : {sum(lens):,} byte  (= kích thước file: {sum(lens) == tong_byte})")

    if so_khoi >= 2 and lens[-1] < lens[0]:
        print("\nCHÍNH XÁC! Bạn đã thấy HDFS chia tệp thành nhiều khối,")
        print("   và khối cuối cùng CHỈ chiếm đúng phần dữ liệu còn lại.")
        print(f"Trong cụm thật, {so_khoi} khối này nằm trên {so_khoi} máy khác nhau")
        print(f"   và {so_khoi} Mapper sẽ chạy SONG SONG — đó là Data Locality.")
    elif so_khoi == 1:
        print("\nLuu y:  Mới có 1 khối. Bạn đã dùng -D dfs.blocksize=1048576 khi -put chưa?")

_kiem_tra_d3()

---
---
# D4. MÔ PHỎNG MAPREDUCE BẰNG PYTHON
### 8 phút ·**TRỌNG TÂM PHẦN THUẬT TOÁN**

### Nhiệm vụ

Viết lại **bằng tay** cả 4 giai đoạn MapReduce ngay trong notebook, để **nhìn thấy dữ liệu biến đổi** qua từng bước. Bước này chưa dùng Hadoop — mục đích là **hiểu thuật toán** trước khi chạy thật ở D5.

```text
① SPLIT    Chia 10.000 dòng thành 3 mảnh          →  3 mảnh
② MAP      Mỗi dòng → cặp <Type, Machine failure> →  10.000 cặp
③ SHUFFLE  Gom các cặp cùng khóa lại              →  3 khóa
④ REDUCE   Tổng hợp mỗi khóa                      →  3 dòng kết quả
```

### Kết quả mong đợi

| Type | Số bản ghi | Số hỏng | Tỷ lệ |
|:---|---:|---:|---:|
| L | 6.000 | 235 | 3,92% |
| M | 2.997 | 83 | 2,77% |
| H | 1.003 | 21 | 2,09% |

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Mô phỏng thuật toán MapReduce bằng Python thuần trên file DATA_PATH,
chia rõ 4 giai đoạn và IN KẾT QUẢ TRUNG GIAN SAU MỖI GIAI ĐOẠN:

1. SPLIT: đọc toàn bộ dòng của file CSV (bỏ dòng tiêu đề), chia thành
   3 mảnh xấp xỉ bằng nhau. In số dòng mỗi mảnh.

2. MAP: viết hàm mapper(mang_dong) nhận danh sách dòng, trả về danh sách
   cặp (Type, Machine_failure). Type là cột thứ 3, Machine failure là cột
   thứ 9 (đánh số từ 1). Áp dụng mapper cho từng mảnh, in số cặp sinh ra.

3. SHUFFLE & SORT: gom tất cả cặp từ 3 mảnh, nhóm theo khóa bằng
   collections.defaultdict(list), rồi sắp xếp theo khóa.
   In các khóa tìm được và số giá trị mỗi khóa.

4. REDUCE: viết hàm reducer(khoa, danh_sach_gia_tri) trả về
   (tổng số bản ghi, số lần hỏng, tỷ lệ phần trăm).
   In bảng kết quả cuối cùng.
```

### Công cụ

`csv.reader` · `collections.defaultdict` · `sorted()` · list comprehension

#### GỢI Ý GIẢI — D4

**Khung code:**
```python
import csv
from collections import defaultdict

# ========== ① SPLIT — chia tách ==========
with open(DATA_PATH, encoding="utf-8") as f:
    tat_ca = list(csv.reader(f))
tieu_de, du_lieu = tat_ca[0], tat_ca[1:]

SO_MANH = 3
kich_thuoc = len(du_lieu) // SO_MANH + 1
manh = [du_lieu[i:i + kich_thuoc] for i in range(0, len(du_lieu), kich_thuoc)]

print("① SPLIT")
for i, m in enumerate(manh):
    print(f"   Mảnh {i}: {len(m):,} dòng")
print(f"   Tổng: {sum(len(m) for m in manh):,} dòng\n")


# ========== ② MAP — ánh xạ ==========
def mapper(mang_dong):
    # Mỗi dòng -> một cặp (Type, Machine failure)
    ket_qua = []
    for cot in mang_dong:
        if len(cot) < 9:
            continue
        ket_qua.append((cot[2], int(cot[8])))    # cột 3 = Type, cột 9 = Machine failure
    return ket_qua

ket_qua_map = [mapper(m) for m in manh]

print("② MAP")
for i, kq in enumerate(ket_qua_map):
    print(f"   Mapper {i} phát ra {len(kq):,} cặp — ví dụ: {kq[:3]}")
tong_cap = sum(len(k) for k in ket_qua_map)
print(f"   Tổng số cặp: {tong_cap:,}\n")


# ========== ③ SHUFFLE & SORT — trộn và sắp xếp ==========
gom = defaultdict(list)
for kq in ket_qua_map:
    for khoa, gia_tri in kq:
        gom[khoa].append(gia_tri)
gom = dict(sorted(gom.items()))

print("③ SHUFFLE & SORT")
for khoa, ds in gom.items():
    print(f"   Khóa '{khoa}' → danh sách {len(ds):,} giá trị: {ds[:5]}...")
print(f"   Từ {tong_cap:,} cặp gom lại còn {len(gom)} khóa\n")


# ========== ④ REDUCE — rút gọn ==========
def reducer(khoa, danh_sach):
    tong = len(danh_sach)
    hong = sum(danh_sach)
    return tong, hong, hong / tong * 100

print("④ REDUCE")
print(f"   {'Type':<6}{'Bản ghi':>10}{'Số hỏng':>10}{'Tỷ lệ':>10}")
print("   " + "-" * 36)
ket_qua_cuoi = {}
for khoa, ds in gom.items():
    tong, hong, ty_le = reducer(khoa, ds)
    ket_qua_cuoi[khoa] = (tong, hong, ty_le)
    print(f"   {khoa:<6}{tong:>10,}{hong:>10,}{ty_le:>9.2f}%")

print(f"\n    {tong_cap:,} cặp  →  {len(ket_qua_cuoi)} dòng kết quả")
print("      Con số này chính là bản chất của phép RÚT GỌN (reduce).")
```

> **Vì sao bước này quan trọng dù chưa dùng Hadoop?**
>
> Vì Hadoop **che giấu** giai đoạn ③ Shuffle & Sort — bạn không viết dòng code nào cho nó, khung Hadoop tự làm. Nếu chưa từng tự viết, bạn sẽ không hiểu vì sao Shuffle lại là giai đoạn **tốn kém nhất** (phải truyền dữ liệu qua mạng để gom các cặp cùng khóa về một chỗ). Đây cũng chính là điều Apache Spark tối ưu ở Buổi 05.

> **Đối chiếu với Pandas cho dễ nhớ:**
>
> | MapReduce | Pandas tương đương |
> |:---|:---|
> | ② Map — phát ra `<Type, failure>` | `df[["Type", "Machine failure"]]` |
> | ③ Shuffle & Sort — gom theo khóa | `.groupby("Type")` |
> | ④ Reduce — tổng hợp mỗi nhóm | `.agg(["size", "sum"])` |

In [ ]:
# --- D4: Mô phỏng MapReduce bằng Python ---
# TODO ①: SPLIT   - đọc CSV, bỏ tiêu đề, chia thành 3 mảnh. In số dòng mỗi mảnh.
# TODO ②: MAP     - hàm mapper() trả về cặp (Type, Machine_failure). In số cặp.
# TODO ③: SHUFFLE - gom theo khóa bằng defaultdict(list), sắp xếp. In các khóa.
# TODO ④: REDUCE  - hàm reducer() tính (tổng, số hỏng, tỷ lệ). In bảng kết quả.
#
# Lưu kết quả cuối vào biến ket_qua_cuoi dạng {Type: (tong, hong, ty_le)}

ket_qua_cuoi = None   # <-- gán kết quả Reduce vào đây

In [ ]:
# =============================================================================
# TỰ KIỂM TRA D4 (không cần sửa)
# =============================================================================
def _kiem_tra_d4():
    CHUAN = {"L": (6000, 235), "M": (2997, 83), "H": (1003, 21)}

    if ket_qua_cuoi is None:
        return print("Luu y:  Bạn chưa gán biến ket_qua_cuoi. Hãy hoàn thành giai đoạn ④ REDUCE.")

    print(f"  {'Type':<6}{'Bản ghi':>10}{'Số hỏng':>10}{'Tỷ lệ':>10}   Đối chiếu")
    print("  " + "-" * 52)
    dat = True
    for t in ["L", "M", "H"]:
        if t not in ket_qua_cuoi:
            print(f"  {t:<6}{'—':>10}{'—':>10}{'—':>10}   [CHUA DAT] thiếu khóa")
            dat = False
            continue
        tong, hong = ket_qua_cuoi[t][0], ket_qua_cuoi[t][1]
        ty_le = hong / tong * 100
        ok = (tong, hong) == CHUAN[t]
        dat &= ok
        print(f"  {t:<6}{tong:>10,}{hong:>10,}{ty_le:>9.2f}%   {'[DAT]' if ok else '[CHUA DAT] lệch'}")
    print("  " + "-" * 52)

    if dat:
        print("\nCHÍNH XÁC! MapReduce mô phỏng cho ra đúng kết quả của Buổi 01.")
        print("Bạn vừa tự tay viết lại điều mà df.groupby('Type') làm trong 1 dòng —")
        print("   nhưng lần này bạn hiểu từng bước bên trong nó.")
    else:
        print("\nKết quả chưa khớp. Kiểm tra: Type là cột thứ 3 (chỉ số 2),")
        print("Machine failure là cột thứ 9 (chỉ số 8), và đã bỏ dòng tiêu đề chưa.")

_kiem_tra_d4()

---
---
# D5. CHẠY MAPREDUCE THẬT TRÊN CỤM HADOOP
### 8 phút ·**TRỌNG TÂM BUỔI HỌC**

### Nhiệm vụ

Viết Mapper và Reducer, chạy qua **Hadoop Streaming** trên dữ liệu đang nằm trên HDFS.

### Vì sao dùng `awk` mà không dùng Python?

Container Hadoop trong khóa học **không cài Python**. Ta dùng **`awk`** — công cụ xử lý văn bản có sẵn trong mọi bản Linux. Điều này lại có lợi: `awk` ngắn gọn nên bạn nhìn thấy ngay bản chất Map và Reduce, không bị phân tâm bởi cú pháp.

### Quy ước bắt buộc của Hadoop Streaming

```text
Dòng dữ liệu ──► [stdin] MAPPER [stdout] ──► "khóa \t giá trị"
                                                    │
                                    Hadoop tự Shuffle & Sort
                                                    │
Kết quả cuối ◄── [stdout] REDUCER [stdin] ◄────────┘
```

Khóa và giá trị **phải ngăn cách bằng ký tự tab** (`\t`). Hadoop tách khóa ở phần trước tab đầu tiên.

### Kết quả mong đợi

```text
Found 2 items
-rw-r--r--   1 root supergroup    0  ... /user/bigdata/lab4/output_mr/_SUCCESS
-rw-r--r--   1 root supergroup   49  ... /user/bigdata/lab4/output_mr/part-00000

H	1003	21	2.09%
L	6000	235	3.92%
M	2997	83	2.77%
```

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
Chạy một job MapReduce thật bằng Hadoop Streaming, dùng hàm hadoop() và các
biến HDFS_IN, HDFS_OUT, STREAM_JAR đã có sẵn:

1. Tạo file /tmp/mapper.sh trong container, nội dung là script bash dùng awk:
   awk -F, '$1 != "UDI" { print $3 "\t" $9 }'
   (bỏ dòng tiêu đề, in cột 3 và cột 9 ngăn cách bằng tab)

2. Tạo file /tmp/reducer.sh trong container, dùng awk với -F"\t":
   cộng dồn tong[$1]++ và hong[$1] += $2, đến END thì in ra
   Type, tổng, số hỏng, tỷ lệ phần trăm

3. chmod +x cho cả hai file

4. QUAN TRỌNG - test thử bằng ống dẫn Linux TRƯỚC khi giao cho Hadoop:
   hdfs dfs -cat <file> | /tmp/mapper.sh | sort | /tmp/reducer.sh

5. Xóa thư mục output cũ bằng hdfs dfs -rm -r -f

6. Chạy job:
   hadoop jar STREAM_JAR -D mapreduce.framework.name=local
     -D mapreduce.job.reduces=1 -files /tmp/mapper.sh,/tmp/reducer.sh
     -input HDFS_IN/ai4i2020.csv -output HDFS_OUT
     -mapper mapper.sh -reducer reducer.sh
   Lọc log để hiển thị các bộ đếm Map input/output records và
   Reduce input/output records.

7. Đọc kết quả: hdfs dfs -ls HDFS_OUT và hdfs dfs -cat HDFS_OUT/part-*
```

### Công cụ

`hadoop jar` · `hadoop-streaming.jar` · `-files` · `-mapper` · `-reducer` · `awk`

#### GỢI Ý GIẢI — D5

**Bước 1 — Tạo Mapper và Reducer trong container:**
```python
MAPPER = r'''#!/bin/bash
# MAP: mỗi dòng CSV -> cặp <Type \t Machine_failure>
awk -F, '$1 != "UDI" { print $3 "\t" $9 }'
'''

REDUCER = r'''#!/bin/bash
# REDUCE: gom theo Type, đếm tổng và số lần hỏng
awk -F"\t" '
{ tong[$1]++; hong[$1] += $2 }
END { for (t in tong) printf "%s\t%d\t%d\t%.2f%%\n", t, tong[t], hong[t], hong[t]*100/tong[t] }
'
'''

# Ghi vào container bằng heredoc trích dẫn (tránh bash diễn giải $ và \)
for ten, noi_dung in [("mapper.sh", MAPPER), ("reducer.sh", REDUCER)]:
    hadoop(f"cat > /tmp/{ten} <<'HET'\n{noi_dung}HET\nchmod +x /tmp/{ten}", hien_thi=False)
    print(f" Đã tạo /tmp/{ten}")
```

**Giải thích `awk` trong Mapper:**

| Thành phần | Ý nghĩa |
|:---|:---|
| `-F,` | Đặt dấu phẩy làm ký tự phân tách cột |
| `$1 != "UDI"` | Bỏ qua dòng tiêu đề (cột 1 của tiêu đề là chữ `UDI`) |
| `$3` | Cột thứ 3 = `Type` → dùng làm **khóa** |
| `$9` | Cột thứ 9 = `Machine failure` → dùng làm **giá trị** |
| `"\t"` | Tab — ký tự Hadoop dùng để tách khóa khỏi giá trị |

**Bước 2 — TEST TRƯỚC KHI GIAO CHO HADOOP** *(mẹo gỡ lỗi quan trọng nhất)*:
```python
print("=== TEST BẰNG ỐNG DẪN LINUX (sort đóng vai Shuffle & Sort) ===")
hadoop(f"hdfs dfs -cat {HDFS_IN}/ai4i2020.csv 2>/dev/null "
       f"| /tmp/mapper.sh | sort | /tmp/reducer.sh")
```
Lệnh này mô phỏng **chính xác** những gì Hadoop làm, nhưng chạy trong 2 giây thay vì 30 giây, và thông báo lỗi dễ đọc hơn nhiều. Nếu bước này sai thì job Hadoop chắc chắn cũng sai.

**Bước 3 — Chạy job thật:**
```python
hdfs(f"-rm -r -f {HDFS_OUT}", hien_thi=False)   # Hadoop từ chối ghi đè output cũ

lenh_job = (
    f"hadoop jar {STREAM_JAR} "
    f"-D mapreduce.framework.name=local "
    f"-D mapreduce.job.reduces=1 "
    f"-files /tmp/mapper.sh,/tmp/reducer.sh "
    f"-input {HDFS_IN}/ai4i2020.csv "
    f"-output {HDFS_OUT} "
    f"-mapper mapper.sh -reducer reducer.sh"
)

print("=== ĐANG CHẠY JOB MAPREDUCE (10-30 giây) ===")
nhat_ky = hadoop(lenh_job + " 2>&1", hien_thi=False)

print("--- BỘ ĐẾM CỦA JOB ---")
for dong in nhat_ky.split("\n"):
    if any(k in dong for k in ["Map input records", "Map output records",
                                "Reduce input records", "Reduce output records",
                                "completed successfully", "Streaming Job Failed"]):
        print("  ", dong.strip())
```

**Bước 4 — Đọc kết quả:**
```python
print("\n=== TỆP ĐẦU RA TRÊN HDFS ===")
hdfs(f"-ls {HDFS_OUT}")

print("\n=== KẾT QUẢ ===")
hdfs(f"-cat {HDFS_OUT}/part-*")
```

---

**Bốn bộ đếm kể trọn câu chuyện MapReduce — hãy tự tìm và giải thích:**

```text
Map input records=10001      ◄── đọc vào 10.001 dòng (kể cả tiêu đề)
Map output records=10000     ◄── phát ra 10.000 cặp (Mapper đã lọc tiêu đề)
Reduce input records=10000   ◄── Reducer nhận đủ, KHÔNG mất dữ liệu trong Shuffle
Reduce output records=3      ◄── kết tinh thành 3 dòng — bản chất phép REDUCE
```

**Hai tệp đầu ra:**

| Tệp | Ý nghĩa |
|:---|:---|
| `_SUCCESS` | Tệp **rỗng**, đánh dấu job hoàn thành trọn vẹn. Các job phía sau kiểm tra tệp này trước khi đọc kết quả |
| `part-00000` | Kết quả do Reducer số 0 ghi ra. Chạy 3 Reducer sẽ có `part-00000`, `part-00001`, `part-00002` |

> **Lỗi hay gặp nhất: `Output directory already exists`**
>
> Hadoop **cố tình** từ chối ghi đè để tránh mất kết quả cũ. Luôn chạy `hdfs dfs -rm -r -f <output>` trước khi chạy lại job.

> **Nếu `part-00000` rỗng:** Mapper không phát ra gì. Quay lại **Bước 2** test bằng ống dẫn Linux — gần như chắc chắn lỗi ở biểu thức `awk`.

In [ ]:
# --- D5: Chạy MapReduce thật bằng Hadoop Streaming ---
# TODO ①: Tạo /tmp/mapper.sh và /tmp/reducer.sh trong container, chmod +x
# TODO ②: TEST TRƯỚC bằng ống dẫn Linux: -cat | mapper.sh | sort | reducer.sh
# TODO ③: Xóa thư mục output cũ bằng hdfs dfs -rm -r -f
# TODO ④: Chạy job bằng hadoop jar STREAM_JAR ... và lọc log lấy bộ đếm
# TODO ⑤: Đọc kết quả bằng -ls và -cat part-*

In [ ]:
# =============================================================================
# TỰ KIỂM TRA D5 (không cần sửa)
# =============================================================================
def _kiem_tra_d5():
    CHUAN = {"L": (6000, 235), "M": (2997, 83), "H": (1003, 21)}

    ds = hadoop(f"hdfs dfs -ls {HDFS_OUT}", hien_thi=False)
    if "part-" not in ds:
        return print(f"Luu y:  Chưa có kết quả tại {HDFS_OUT}. Hãy hoàn thành bước ④.")

    print("Tệp đầu ra trên HDFS:")
    print(f"    _SUCCESS   : {'[DAT] có' if '_SUCCESS' in ds else '[CHUA DAT] thiếu (job chưa hoàn tất?)'}")
    print(f"    part-00000 : {'[DAT] có' if 'part-00000' in ds else '[CHUA DAT] thiếu'}")

    noi_dung = hadoop(f"hdfs dfs -cat {HDFS_OUT}/part-*", hien_thi=False)
    doc = {}
    for d in noi_dung.strip().split("\n"):
        p = d.split("\t")
        if len(p) >= 3:
            doc[p[0].strip()] = (int(p[1]), int(p[2]))

    print(f"\n  {'Type':<6}{'Bản ghi':>10}{'Số hỏng':>10}{'Tỷ lệ':>10}   Đối chiếu Buổi 01")
    print("  " + "-" * 60)
    dat = True
    for t in ["L", "M", "H"]:
        if t not in doc:
            print(f"  {t:<6}{'—':>10}{'—':>10}{'—':>10}   [CHUA DAT] thiếu")
            dat = False
            continue
        tong, hong = doc[t]
        ok = (tong, hong) == CHUAN[t]
        dat &= ok
        print(f"  {t:<6}{tong:>10,}{hong:>10,}{hong/tong*100:>9.2f}%   {'[DAT] khớp' if ok else '[CHUA DAT] lệch'}")
    print("  " + "-" * 60)

    if dat and "_SUCCESS" in ds:
        print("""
 XUẤT SẮC! Bạn vừa chạy một job MapReduce THẬT trên cụm Hadoop.

   Ba con số này giống hệt kết quả df.groupby('Type') ở Buổi 01 —
   nhưng lần này chúng được tính bằng một mô hình có thể mở rộng ra
   hàng trăm máy, và tự chạy lại phần việc nếu một máy chết giữa chừng.

    Câu hỏi cho bước D6: nếu Pandas nhanh hơn nhiều, vì sao còn cần Hadoop?""")
    else:
        print("\nKết quả chưa khớp. Test lại mapper/reducer bằng ống dẫn Linux:")
        print(f"   hdfs dfs -cat {HDFS_IN}/ai4i2020.csv | /tmp/mapper.sh | sort | /tmp/reducer.sh")

_kiem_tra_d5()

---
# D6. ĐỐI CHIẾU MAPREDUCE VỚI PANDAS
### 4 phút

### Nhiệm vụ

Chạy lại `groupby` bằng Pandas trên chính tệp đó, so sánh kết quả và **thời gian chạy**.

### PROMPT CHO AI AGENT — *dán nguyên khối này*

```
So sánh MapReduce với Pandas trên cùng bài toán:
1. Đọc DATA_PATH bằng pandas, đo thời gian bằng time.perf_counter()
2. Chạy df.groupby("Type").agg() tính số bản ghi, số lần hỏng, tỷ lệ hỏng,
   đo thời gian riêng cho phép groupby
3. In bảng kết quả Pandas
4. In bảng so sánh: thời gian Pandas với thời gian job MapReduce (khoảng 10-30 giây)
5. Kết luận về việc khi nào nên dùng công cụ nào
```

### Công cụ

`pandas.read_csv()` · `.groupby().agg()` · `time.perf_counter()`

#### GỢI Ý GIẢI — D6

**Khung code:**
```python
import time
import pandas as pd

t0 = time.perf_counter()
df = pd.read_csv(DATA_PATH)
t_doc = time.perf_counter() - t0

t0 = time.perf_counter()
kq_pandas = df.groupby("Type").agg(
    so_ban_ghi=("Machine failure", "size"),
    so_lan_hong=("Machine failure", "sum"),
).assign(ty_le_hong=lambda d: d.so_lan_hong / d.so_ban_ghi * 100)
t_groupby = time.perf_counter() - t0

print(kq_pandas.round(2))
print(f"\nĐọc file  : {t_doc*1000:.1f} ms")
print(f"GroupBy   : {t_groupby*1000:.1f} ms")
print(f"Tổng cộng : {(t_doc + t_groupby)*1000:.1f} ms")
t_pandas_ms = (t_doc + t_groupby) * 1000
print(f"\nJob MapReduce vừa rồi mất khoảng 10.000-30.000 ms")
print(f"→ Pandas nhanh hơn khoảng {10000/t_pandas_ms:,.0f} đến "
      f"{30000/t_pandas_ms:,.0f} lần trên bộ dữ liệu này")
```

---

### CÂU HỎI QUAN TRỌNG NHẤT BUỔI HỌC

> **"Nếu Pandas nhanh hơn hàng nghìn lần, vì sao còn cần Hadoop?"**

Hãy tự trả lời trước, rồi mới đọc bảng dưới.

| Tiêu chí | Pandas | Hadoop MapReduce |
|:---|:---|:---|
| Dữ liệu 10.000 dòng | Mili giây | Hàng chục giây — thừa thãi |
| Dữ liệu 10 TB | **Không chạy nổi**, tràn RAM | Chạy được, chỉ cần thêm máy |
| Giới hạn kích thước | RAM của **một** máy | Tổng dung lượng **cả cụm** |
| Một máy chết giữa chừng | Mất trắng, chạy lại từ đầu | Tự chạy lại phần việc trên máy khác |
| Chi phí khởi động | Gần như bằng 0 | Hàng chục giây |
| Độ phức tạp khi viết | Một dòng `groupby` | Hai hàm + cấu hình job |

**Kết luận cần chốt:**

> Hadoop **không nhanh hơn** Pandas. Nó **chạy được ở quy mô mà Pandas không chạy nổi**, và **không chết khi phần cứng chết**.
>
> Chọn công cụ theo **quy mô dữ liệu**, không theo mức độ "hiện đại" của công nghệ. Dùng Hadoop cho 10.000 dòng là sai lầm nghề nghiệp — giống như thuê xe tải chở một thùng sữa.

In [ ]:
# --- D6: Đối chiếu MapReduce với Pandas ---
# TODO: Đọc DATA_PATH bằng pandas, đo thời gian
# TODO: groupby("Type") tính số bản ghi, số hỏng, tỷ lệ — đo thời gian
# TODO: In bảng kết quả và so sánh thời gian với job MapReduce

#### Câu trả lời của bạn — D6

**Theo bạn, khi nào nên dùng Hadoop MapReduce thay vì Pandas?**

*(Viết câu trả lời tại đây. Hãy nêu ít nhất hai tình huống cụ thể trong bối cảnh nhà máy ở bài toán này — ví dụ: dữ liệu tích lũy sau bao lâu thì Pandas không kham nổi?)*

**Ngược lại, dùng Hadoop khi nào là sai lầm?**

*(Viết câu trả lời tại đây.)*

---
---
# TỔNG KẾT BUỔI HỌC

## Sản phẩm phải nộp

| # | Tệp | Nội dung |
|:--|:---|:---|
| 1 | `Buoi_04_Hadoop_Student.ipynb` | Notebook đã chạy hết, không còn ô TODO trống |
| 2 | `outputs/report_buoi04.md` | Báo cáo các lệnh đã chạy và kết quả |
| 3 | Ảnh chụp `localhost:9870` | Tab **Overview** và tab **Browse the file system** |

## Tiêu chí hoàn thành

- [ ] Cụm HDFS báo cáo `Live datanodes (1)`, mở được `localhost:9870`
- [ ] Đẩy thành công `ai4i2020.csv` lên HDFS, `wc -l` ra đúng **10001**
- [ ] Dùng `fsck` chứng minh tệp 4,9 MB chia thành **5 khối**, khối cuối không đầy
- [ ] Mô phỏng MapReduce bằng Python, in kết quả trung gian đủ **4 giai đoạn**
- [ ] Job Hadoop Streaming chạy thật, có `_SUCCESS` và `part-00000`
- [ ] Kết quả **trùng khớp Buổi 01**: L 3,92% · M 2,77% · H 2,09%
- [ ] Giải thích được 4 bộ đếm Map/Reduce input/output records

## Bảng chấm điểm

| Tiêu chí | Điểm |
|:---|---:|
| D1–D2 — Thao tác HDFS CLI thành thạo, dữ liệu lên đúng vị trí | 20 |
| D3 — Chứng minh và giải thích được cơ chế chia khối | 20 |
| D4 — Mô phỏng đúng 4 giai đoạn, có in kết quả trung gian | 20 |
| D5 — Job MapReduce thật chạy thành công, đọc được bộ đếm | 25 |
| D6 — So sánh có lý lẽ, trả lời được "vì sao cần Hadoop" | 15 |
| **Tổng** | **100** |

---

## Lỗi thường gặp và cách xử lý

| Lỗi | Nguyên nhân | Cách xử lý |
|:---|:---|:---|
| `Output directory already exists` | Hadoop từ chối ghi đè kết quả cũ | `hdfs dfs -rm -r -f <output>` trước khi chạy lại |
| `put: File exists` | Tệp đã có trên HDFS | Thêm cờ `-f`: `hdfs dfs -put -f ...` |
| `cat: Unable to write to output stream` | `head` đóng ống dẫn sớm | **Không phải lỗi** — bỏ qua |
| `Permission denied` khi chạy mapper | Script chưa có quyền thực thi | `chmod +x /tmp/mapper.sh /tmp/reducer.sh` |
| `python3: command not found` | Container Hadoop không cài Python | Dùng `awk` như hướng dẫn (xem Phụ lục B tài liệu .docx) |
| `Name node is in safe mode` | NameNode vừa khởi động, đang tự kiểm tra | Đợi 30–60 giây, hoặc `hdfs dfsadmin -safemode leave` |
| `Connection refused` cổng 9870 | Container chưa chạy xong | `docker compose ps`, đợi trạng thái `healthy` |
| `part-00000` rỗng | Mapper không phát ra gì | Test riêng: `hdfs dfs -cat file \| /tmp/mapper.sh \| head` |

> **Mẹo gỡ lỗi quan trọng nhất của buổi học:**
>
> **Luôn test Mapper và Reducer bằng ống dẫn Linux TRƯỚC khi giao cho Hadoop.**
> ```
> hdfs dfs -cat <file> | /tmp/mapper.sh | sort | /tmp/reducer.sh
> ```
> Lệnh này mô phỏng chính xác những gì Hadoop làm (`sort` đóng vai Shuffle & Sort), nhưng chạy trong 2 giây thay vì 30 giây và báo lỗi dễ đọc hơn nhiều.

---

## Buổi tiếp theo

| Buổi | Chủ đề | Liên hệ với hôm nay |
|:---|:---|:---|
| **05** | Apache Spark | Chạy lại **chính phép tính này** bằng Spark, đo xem nhanh hơn bao nhiêu lần. Spark giữ dữ liệu trung gian **trong RAM** thay vì ghi xuống đĩa sau mỗi giai đoạn — đó là lý do nó nhanh hơn MapReduce |
| **06** | Data Wrangling | Đọc dữ liệu từ HDFS thay vì tệp cục bộ |
| **13** | Học máy | Bộ đặc trưng huấn luyện lưu trên HDFS ở định dạng Parquet |

> Hôm nay bạn thấy Shuffle & Sort phải ghi dữ liệu xuống đĩa giữa hai giai đoạn. Hãy nhớ điều đó — Buổi 05 sẽ cho bạn thấy Spark loại bỏ bước ghi đĩa ấy như thế nào, và nhanh hơn bao nhiêu lần.

In [ ]:
# =============================================================================
#  NGHIỆM THU CUỐI BUỔI - Chạy trước khi nộp bài (không cần sửa)
# =============================================================================
print("=" * 74)
print("BẢNG NGHIỆM THU BUỔI 04".center(74))
print("=" * 74)

muc = []
_g = globals()

# D1
_bc = hadoop("hdfs dfsadmin -report", hien_thi=False)
muc.append(("D1  Cụm HDFS hoạt động, có DataNode sống", "Live datanodes (1)" in _bc))

# D2
_ls_in = hadoop(f"hdfs dfs -ls {HDFS_IN}", hien_thi=False)
muc.append(("D2  Đã đẩy ai4i2020.csv lên HDFS", "ai4i2020.csv" in _ls_in))
_n = hadoop(f"hdfs dfs -cat {HDFS_IN}/ai4i2020.csv 2>/dev/null | wc -l", hien_thi=False).strip()
muc.append((f"D2  Dữ liệu nguyên vẹn 10001 dòng (được {_n or '—'})", _n == "10001"))

# D3
import re as _re
_fs = hadoop(f"hdfs fsck {HDFS_IN}/ai4i_big.csv -files -blocks", hien_thi=False)
_lens = [int(x) for x in _re.findall(r"len=(\d+)", _fs)]
muc.append((f"D3  Chứng minh chia nhiều khối (được {len(_lens)} khối)", len(_lens) >= 2))
muc.append(("D3  Khối cuối không bị lấp đầy", len(_lens) >= 2 and _lens[-1] < _lens[0]))

# D4
_kq = _g.get("ket_qua_cuoi")
muc.append(("D4  Mô phỏng MapReduce cho kết quả đúng",
            isinstance(_kq, dict) and
            all(t in _kq and _kq[t][0] == n and _kq[t][1] == h
                for t, (n, h) in {"L": (6000, 235), "M": (2997, 83), "H": (1003, 21)}.items())))

# D5
_ls_out = hadoop(f"hdfs dfs -ls {HDFS_OUT}", hien_thi=False)
muc.append(("D5  Job MapReduce có tệp _SUCCESS", "_SUCCESS" in _ls_out))
_out = hadoop(f"hdfs dfs -cat {HDFS_OUT}/part-*", hien_thi=False)
_doc = {}
for _d in _out.strip().split("\n"):
    _p = _d.split("\t")
    if len(_p) >= 3:
        _doc[_p[0].strip()] = (int(_p[1]), int(_p[2]))
muc.append(("D5  Kết quả MapReduce khớp Buổi 01 (L/M/H)",
            _doc == {"L": (6000, 235), "M": (2997, 83), "H": (1003, 21)}))

dat = 0
for ten, ok in muc:
    print(f"  {'[DAT]' if ok else '[    ]'}  {ten}")
    dat += bool(ok)

print("-" * 74)
print(f"HOÀN THÀNH: {dat}/{len(muc)} mục  ({dat/len(muc)*100:.0f}%)")
print("=" * 74)
if dat == len(muc):
    print("Chúc mừng! Bạn đã hoàn thành trọn vẹn Buổi 04.")
    print("Bạn đã lưu dữ liệu trên hệ thống tệp phân tán và tính toán bằng")
    print("MapReduce — hai nền tảng của toàn bộ hệ sinh thái Big Data.")
else:
    print("Còn mục chưa xong (ô trống [    ]). Hoàn thiện rồi chạy lại ô này.")
print("=" * 74)